In [1]:
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
file_path = '/Users/adam/Projects/FYP/DATA2024.xlsx'

In [3]:
raw = pd.read_excel(file_path)

In [4]:
data = pd.DataFrame(raw)

In [5]:
def sort_key(value):
    """
    Returns a tuple (priority, original_value) for sorting.
    Priority:
        0 -> starts with a digit
        1 -> starts with a Latin letter (A-Z, a-z)
        2 -> starts with a Chinese character (CJK Unified Ideographs block)
        3 -> everything else (including empty strings)
    """
    if not isinstance(value, str) or len(value) == 0:
        return (3, str(value))          # empty or non‑strings go last

    first_char = value[0]

    # Digits 0-9
    if first_char.isdigit():
        return (0, value)

    # Latin letters (basic A-Z, a-z)
    if 'a' <= first_char.lower() <= 'z':
        return (1, value)

    # Chinese characters (main CJK block)
    if 0x4E00 <= ord(first_char) <= 0x9FFF:
        return (2, value)

    # Fallback for any other character
    return (3, value)

In [6]:
clean_data = data[data[' 狀態'] != '已傳真']
clean_data1 = clean_data.iloc[:,np.r_[0:17, 21]]
clean_data1

,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,床號/備註,設備,預定,下單,派遣,報到,完成,工友
0,79241,,完成,即時,預定,S,送3F X光,7H,3FXRAY,,,大床,2024-01-02 14:00:00,2024-01-01 10:28:24,2024-01-02 14:35:48,2024-01-02 14:38:15,2024-01-02 14:44:57,"0030,0090"
1,79264,,完成,即時,預定,S,送病人,5L,5L,2F 回床,接觸傳染,大床O2,2024-01-05 10:00:00,2024-01-01 15:30:05,2024-01-05 10:35:39,2024-01-05 10:41:25,2024-01-05 11:08:53,"3023,3047"
2,79274,,完成,超緊急,安排,S,送標本,抽血室,化驗室,,如需送往其他醫院，請於此註明,無工具,,2024-01-02 10:01:00,2024-01-02 10:01:46,2024-01-02 10:04:34,2024-01-02 10:09:12,3048
3,79275,,完成,超緊急,安排,S,送標本,抽血室,化驗室,,,無工具,,2024-01-02 08:16:00,2024-01-02 08:22:12,2024-01-02 08:24:36,2024-01-02 08:28:13,0073
4,79276,79516(2),完成,超緊急,安排,S,送標本,抽血室,化驗室,,,無工具,,2024-01-02 11:31:00,2024-01-02 11:34:31,2024-01-02 11:35:16,2024-01-02 11:42:23,3026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84376,97352,,完成,超緊急,普通,S,NEATS病人,ENQ,6H,,車牌3934,輪床,,2024-12-31 17:24:06,2024-12-31 17:25:38,2024-12-31 17:32:35,2024-12-31 17:46:35,3026
84377,97354,,完成,超緊急,普通,S,送入院,ENQ,5L,,NEATS LICENSE 3934,輪床,,2024-12-31 17:27:14,2024-12-31 17:31:45,2024-12-31 17:34:14,2024-12-31 17:54:17,3048
84378,97355,,完成,超緊急,普通,S,送入院,ENQ,6L,,NEATS LICENSE NO;3934,無工具,,2024-12-31 17:28:59,2024-12-31 17:33:02,2024-12-31 17:33:12,2024-12-31 17:54:39,0009
84379,97356,,完成,即時,普通,S,送文件,3B,2B,,,無工具,,2024-12-31 17:36:39,2024-12-31 17:37:21,2024-12-31 17:41:54,2024-12-31 17:43:58,3029


In [7]:
# Low-overhead services only: object deliveries with minimal pickup/dropoff time.
# Excluded: 送空瓶/箱 (multi-stop), 出/轉/覆 (patient transfer), 雪藥來回 (round-trip),
#           抽血急標本 (blood draw), 文件簽收 (signature wait), 送加餐 (food),
#           運送排版/送UVDI-360/送醫療器材 (equipment setup), 取文件 (pickup wait)
LOW_OVERHEAD_SERVICES = [
    '送標本', '送文件', '送X光', '送X光片', '送3F X光',
    '送藥', '送培養', '送新冠標本', 'TKO送培/血', '送輕型物件'
]

main_data = clean_data1[clean_data1['設備'] == '無工具']
main_data = main_data[main_data['主編號(序)'] == ' ']
main_data = main_data[main_data['途經'] == ' ']
main_data = main_data[main_data[' 從 '] != main_data[' 往']]
main_data = main_data[main_data['服務'].isin(LOW_OVERHEAD_SERVICES)]
comment = sorted(main_data['床號/備註'].unique(), key=sort_key)
main_data

,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,床號/備註,設備,預定,下單,派遣,報到,完成,工友
2,79274,,完成,超緊急,安排,S,送標本,抽血室,化驗室,,如需送往其他醫院，請於此註明,無工具,,2024-01-02 10:01:00,2024-01-02 10:01:46,2024-01-02 10:04:34,2024-01-02 10:09:12,3048
3,79275,,完成,超緊急,安排,S,送標本,抽血室,化驗室,,,無工具,,2024-01-02 08:16:00,2024-01-02 08:22:12,2024-01-02 08:24:36,2024-01-02 08:28:13,0073
5,79277,,完成,超緊急,安排,S,送標本,抽血室,化驗室,,,無工具,,2024-01-02 12:46:00,2024-01-02 12:48:31,2024-01-02 12:50:30,2024-01-02 12:52:47,0071
27,79312,,完成,緊急,普通,S,送文件,1C,HIRD,,,無工具,,2024-01-02 08:48:34,2024-01-02 08:49:30,2024-01-02 08:50:47,2024-01-02 08:54:30,0030
28,79313,,完成,緊急,普通,S,送標本,3B,化驗室,,如需送往其他醫院，請於此註明,無工具,,2024-01-02 08:51:59,2024-01-02 08:53:31,2024-01-02 08:57:58,2024-01-02 09:00:06,3023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84361,97326,,完成,緊急,普通,S,送文件,門診部,PRC,,,無工具,,2024-12-31 16:28:58,2024-12-31 16:40:06,2024-12-31 16:41:05,2024-12-31 16:43:04,0008
84362,97327,,完成,緊急,普通,S,送文件,PRC,3B,,,無工具,,2024-12-31 16:31:28,2024-12-31 16:35:48,2024-12-31 16:36:33,2024-12-31 16:38:12,0008
84363,97328,,完成,緊急,普通,S,送標本,5H,化驗室,,如需送往其他醫院，請於此註明,無工具,,2024-12-31 16:34:05,2024-12-31 16:36:00,2024-12-31 16:39:09,2024-12-31 16:46:05,0035
84371,97344,,完成,即時,預定,S,送文件,1B,1FMSW,,,無工具,2025-01-02 09:00:00,2024-12-31 17:05:22,2025-01-02 09:03:46,2025-01-02 09:05:32,2025-01-02 09:10:26,0035


In [8]:
import re

# Define patterns and keywords for unwanted content
room_pattern = re.compile(r'\b\d+[A-Za-z]?\b|\b[A-Za-z]+\d+\b|#\d+|bed\s*#?\d+|[0-9]+[A-Za-z]+床位?', re.IGNORECASE)
place_keywords = [
    '本院', '將軍澳醫院', '聯合醫院', '主座', '日間醫院', '職業治療部', '工程部', '抽血服務室',
    '門診', 'lab', '化驗室', '消毒房', '物料部', '總務處', '繳費處', '膳食部', '藥房', '行政部',
    '被服房', '詢問處', '課室', '財務部', '院牧部', '電腦部', '飯堂', 'CDU', 'SOPD', 'TKO', 'G/F',
    '走廊', '附', 'HIRD', 'ICSC', 'ICT', 'MED', 'NSD', 'OT', 'P&PC', 'PMM', 'PRC', 'PRO', 'PT', 'SDU',
    'X光部', '保安室', '卸貨區', '圖書館', '垃圾房', '大堂', '太平間', '家居', '專職2F', '支援部',
    '標本室', '活動室', '疥瘡房', '陳笑風','院1C, LAB'
]
route_keywords = [
    '行round', '到達', '往', '從', 'to', 'from', '起點', '終點', 
    '請於', '前到達',  '送回', '歸還', '借用時間', '非辦公時間', '通知總機'
]
heavy_keywords = [
    '輪床', '屏風', '藥車', '氧氣樽', '角度床', '超聲波機', 'UV機', 'bladderscan', 'infusion pump',
    '暖風機', '污衣桶', '垃圾桶', '手提電腦', '排版車', '牌版車', '急救公仔', '殘肢', '氧氣樽',
    '輪椅', '吊機', '血糖機',  
    '黃袋', '抽血帶', 'bed', 'Bed', 'BED',
    '旦白寶', '黃紙', '透明大膠箱',
    '入院牌板袋', '空床', '酒精', '床','機'
]

def is_noisy(comment):
    """Return True if comment contains room/place/route/heavy references."""
    if not isinstance(comment, str):
        return False
    # Check room number patterns
    if room_pattern.search(comment):
        return True
    # Check place keywords
    if any(kw in comment for kw in place_keywords):
        return True
    # Check route keywords
    if any(kw in comment for kw in route_keywords):
        return True
    # Check heavy items (optional – if you want to keep heavy items as simple, remove this)
    if any(kw in comment for kw in heavy_keywords):
        return True
    # Additional: if comment contains a time pattern like "9:05"
    if re.search(r'\d{1,2}:\d{2}', comment):
        return True
    return False



# Filter out noisy comments, keep clean ones
clean_comments = [c for c in comment if not is_noisy(c)]
"""
# Print or use the result
for c in clean_comments:
    print(c)
"""
print(type(clean_comments))


<class 'list'>


In [9]:
def format_timedelta_robust(td):
    total_seconds = td.total_seconds()
    sign = '-' if total_seconds < 0 else ''
    
    total_seconds = abs(total_seconds)
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = int(total_seconds % 60)
    
    return f"{sign}{hours:02d}:{minutes:02d}:{seconds:02d}"

In [10]:
main_data = main_data[main_data['床號/備註'].isin(clean_comments)].copy()
main_data['Traveling_Time'] = main_data['完成'] - main_data['報到']
main_data['Traveling_Time'] = main_data['Traveling_Time'].apply(format_timedelta_robust)
main_data['Traveling_Time'] = pd.to_timedelta(main_data['Traveling_Time'])

# Drop rows with non-positive travel times (data entry errors)
main_data = main_data[main_data['Traveling_Time'] > pd.Timedelta(0)]

main_data['route_pair'] = main_data.apply(
    lambda row: tuple(sorted([row[' 從 '], row[' 往']])), axis=1
)

# Minimum sample size: require >= 5 observations per route pair
route_counts = main_data.groupby('route_pair').size()
valid_routes = route_counts[route_counts >= 5].index
main_data = main_data[main_data['route_pair'].isin(valid_routes)]

print(f'Route pairs with >= 5 observations: {len(valid_routes)}')

# Use 75th percentile — median had ~7.5-min std dev per route pair; p75 builds in a buffer
# to support the 15-minute KPI without requiring stochastic solver input
median_per_route = main_data.groupby('route_pair')['Traveling_Time'].quantile(0.75)
main_data['Median_Traveling_Time'] = main_data['route_pair'].map(median_per_route)

# Build stats dataframe for the report sheet in travel_times.xlsx
_stats_agg = main_data.groupby('route_pair')['Traveling_Time']
stats_df = pd.DataFrame({
    'n': _stats_agg.count(),
    'median_min': _stats_agg.median().dt.total_seconds() / 60,
    'p75_min': _stats_agg.quantile(0.75).dt.total_seconds() / 60,
    'std_min': _stats_agg.std().dt.total_seconds() / 60,
}).reset_index()
stats_df[['from', 'to']] = pd.DataFrame(stats_df['route_pair'].tolist(), index=stats_df.index)
stats_df = stats_df[['from', 'to', 'n', 'median_min', 'p75_min', 'std_min']].sort_values(['from', 'to']).reset_index(drop=True)

main_data

Route pairs with >= 5 observations: 177


,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,...,設備,預定,下單,派遣,報到,完成,工友,Traveling_Time,route_pair,Median_Traveling_Time
3,79275,,完成,超緊急,安排,S,送標本,抽血室,化驗室,,...,無工具,,2024-01-02 08:16:00,2024-01-02 08:22:12,2024-01-02 08:24:36,2024-01-02 08:28:13,0073,0 days 00:03:37,"(化驗室, 抽血室)",0 days 00:03:30.750000
5,79277,,完成,超緊急,安排,S,送標本,抽血室,化驗室,,...,無工具,,2024-01-02 12:46:00,2024-01-02 12:48:31,2024-01-02 12:50:30,2024-01-02 12:52:47,0071,0 days 00:02:17,"(化驗室, 抽血室)",0 days 00:03:30.750000
27,79312,,完成,緊急,普通,S,送文件,1C,HIRD,,...,無工具,,2024-01-02 08:48:34,2024-01-02 08:49:30,2024-01-02 08:50:47,2024-01-02 08:54:30,0030,0 days 00:03:43,"(1C, HIRD)",0 days 00:05:42
33,79318,,完成,即時,普通,S,送文件,HIRD,A&D,,...,無工具,,2024-01-02 08:56:11,2024-01-02 08:58:56,2024-01-02 09:00:32,2024-01-02 09:04:19,3022,0 days 00:03:47,"(A&D, HIRD)",0 days 00:05:24
34,79319,,完成,緊急,普通,S,送文件,7H,藥房,,...,無工具,,2024-01-02 08:59:00,2024-01-02 09:01:20,2024-01-02 09:07:02,2024-01-02 09:11:07,3026,0 days 00:04:05,"(7H, 藥房)",0 days 00:05:36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84342,97299,,完成,即時,普通,S,送文件,6L,HIRD,,...,無工具,,2024-12-31 15:54:54,2024-12-31 15:55:05,2024-12-31 15:58:53,2024-12-31 16:02:53,3027,0 days 00:04:00,"(6L, HIRD)",0 days 00:04:10
84349,97311,,完成,即時,普通,S,送文件,6H,X光部,,...,無工具,,2024-12-31 16:02:40,2024-12-31 16:03:17,2024-12-31 16:05:54,2024-12-31 16:14:23,3027,0 days 00:08:29,"(6H, X光部)",0 days 00:07:23
84361,97326,,完成,緊急,普通,S,送文件,門診部,PRC,,...,無工具,,2024-12-31 16:28:58,2024-12-31 16:40:06,2024-12-31 16:41:05,2024-12-31 16:43:04,0008,0 days 00:01:59,"(PRC, 門診部)",0 days 00:03:06
84362,97327,,完成,緊急,普通,S,送文件,PRC,3B,,...,無工具,,2024-12-31 16:31:28,2024-12-31 16:35:48,2024-12-31 16:36:33,2024-12-31 16:38:12,0008,0 days 00:01:39,"(3B, PRC)",0 days 00:02:43.250000


In [11]:
"""
main_more30 = main_data.copy()

route_counts = main_more30.groupby('route_pair')['route_pair'].transform('size')
main_more30 = main_more30[route_counts > 30]
"""

"\nmain_more30 = main_data.copy()\n\nroute_counts = main_more30.groupby('route_pair')['route_pair'].transform('size')\nmain_more30 = main_more30[route_counts > 30]\n"

In [12]:
"""# List of locations to filter
locations_of_interest = ['HIRD', '化驗室', '膳食部', '門診部']

# Create a mask: True if either element of the tuple is in the list
mask = main_more30['route_pair'].apply(
    lambda pair: any(loc in pair for loc in locations_of_interest)
)

# Apply the mask
filtered_routes = main_more30[mask]

# Optional: keep only unique route pairs and their median times
unique_routes = filtered_routes[['route_pair', 'Median_Traveling_Time']].drop_duplicates()
unique_routes
"""

"# List of locations to filter\nlocations_of_interest = ['HIRD', '化驗室', '膳食部', '門診部']\n\n# Create a mask: True if either element of the tuple is in the list\nmask = main_more30['route_pair'].apply(\n    lambda pair: any(loc in pair for loc in locations_of_interest)\n)\n\n# Apply the mask\nfiltered_routes = main_more30[mask]\n\n# Optional: keep only unique route pairs and their median times\nunique_routes = filtered_routes[['route_pair', 'Median_Traveling_Time']].drop_duplicates()\nunique_routes\n"

In [13]:
routes = main_data[['route_pair', 'Median_Traveling_Time']].drop_duplicates()
routes

,route_pair,Median_Traveling_Time
3,"(化驗室, 抽血室)",0 days 00:03:30.750000
27,"(1C, HIRD)",0 days 00:05:42
33,"(A&D, HIRD)",0 days 00:05:24
34,"(7H, 藥房)",0 days 00:05:36
104,"(PT, X光部)",0 days 00:03:46
...,...,...
46331,"(3B, PRC)",0 days 00:02:43.250000
51719,"(3L, 5L)",0 days 00:07:50.750000
56104,"(2FDOM, 5L)",0 days 00:05:38.750000
67768,"(6L, 7H)",0 days 00:05:16


In [14]:
輪椅_data = clean_data1[(clean_data1['設備'] == '輪椅') | (clean_data1['設備'] == '輪椅O2')]
輪椅_data = 輪椅_data[輪椅_data['主編號(序)'] == ' ']
輪椅_data = 輪椅_data[輪椅_data['途經'] == ' ']
輪椅_data = 輪椅_data[輪椅_data[' 從 '] != 輪椅_data[' 往']]
輪椅_data = 輪椅_data[輪椅_data['服務'].isin(LOW_OVERHEAD_SERVICES)]
comment = sorted(輪椅_data['床號/備註'].unique(), key=sort_key)

輪椅_data = 輪椅_data[輪椅_data['床號/備註'].isin(clean_comments)].copy()
輪椅_data['Traveling_Time'] = 輪椅_data['完成'] - 輪椅_data['報到']
輪椅_data['Traveling_Time'] = 輪椅_data['Traveling_Time'].apply(format_timedelta_robust)
輪椅_data['Traveling_Time'] = pd.to_timedelta(輪椅_data['Traveling_Time'])
輪椅_data = 輪椅_data[輪椅_data['Traveling_Time'] > pd.Timedelta(0)]

輪椅_data['route_pair'] = 輪椅_data.apply(
    lambda row: tuple(sorted([row[' 從 '], row[' 往']])), axis=1
)
median_per_route_輪椅 = 輪椅_data.groupby('route_pair')['Traveling_Time'].quantile(0.75)
輪椅_data['Median_Traveling_Time'] = 輪椅_data['route_pair'].map(median_per_route_輪椅)
輪椅_data

,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,...,設備,預定,下單,派遣,報到,完成,工友,Traveling_Time,route_pair,Median_Traveling_Time
46,79331,,完成,緊急,普通,S,送X光,2A,X光部,,...,輪椅,,2024-01-02 09:43:39,2024-01-02 10:49:08,2024-01-02 11:00:04,2024-01-02 11:04:02,0002,0 days 00:03:58,"(2A, X光部)",0 days 00:04:29
88,79377,,完成,即時,普通,S,送X光,3E,X光部,,...,輪椅,,2024-01-02 09:25:17,2024-01-02 09:40:01,2024-01-02 09:43:14,2024-01-02 09:45:34,0035,0 days 00:02:20,"(3E, X光部)",0 days 00:03:19
89,79378,,完成,即時,普通,S,送X光,3E,X光部,,...,輪椅,,2024-01-02 09:25:23,2024-01-02 09:56:28,2024-01-02 09:56:40,2024-01-02 10:00:28,3026,0 days 00:03:48,"(3E, X光部)",0 days 00:03:19
91,79380,,完成,即時,普通,S,送X光,3E,X光部,,...,輪椅,,2024-01-02 09:28:21,2024-01-02 09:57:46,2024-01-02 10:03:21,2024-01-02 10:06:41,0071,0 days 00:03:20,"(3E, X光部)",0 days 00:03:19
92,79381,,完成,即時,普通,S,送X光,3E,X光部,,...,輪椅,,2024-01-02 09:28:24,2024-01-02 09:59:08,2024-01-02 10:01:04,2024-01-02 10:05:29,0029,0 days 00:04:25,"(3E, X光部)",0 days 00:03:19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84116,96973,,完成,即時,普通,S,送X光,3E,X光部,,...,輪椅,,2024-12-31 09:47:30,2024-12-31 10:00:03,2024-12-31 10:02:33,2024-12-31 10:07:26,0037,0 days 00:04:53,"(3E, X光部)",0 days 00:03:19
84120,96977,,完成,即時,普通,S,送X光,3E,X光部,,...,輪椅,,2024-12-31 09:50:14,2024-12-31 10:02:25,2024-12-31 10:06:24,2024-12-31 10:11:40,3048,0 days 00:05:16,"(3E, X光部)",0 days 00:03:19
84141,97004,,完成,緊急,普通,S,送X光,1B,X光部,,...,輪椅,,2024-12-31 10:15:14,2024-12-31 10:30:17,2024-12-31 10:35:16,2024-12-31 10:37:12,3023,0 days 00:01:56,"(1B, X光部)",0 days 00:03:21.500000
84292,97225,,完成,緊急,普通,S,送3F X光,3E,3FXRAY,,...,輪椅,,2024-12-31 14:41:10,2024-12-31 14:51:54,2024-12-31 14:55:24,2024-12-31 15:04:39,0025,0 days 00:09:15,"(3E, 3FXRAY)",0 days 00:07:18.500000


In [15]:
routes_輪椅 = routes.copy()
routes_輪椅['Median_Traveling_Time_輪椅'] = routes_輪椅['route_pair'].map(median_per_route_輪椅)
routes_輪椅 = routes_輪椅[routes_輪椅['Median_Traveling_Time_輪椅'].notna()]
routes_輪椅['Time_diff_輪椅'] = routes_輪椅['Median_Traveling_Time_輪椅'] - routes_輪椅['Median_Traveling_Time']
輪椅_pre_time = routes_輪椅['Time_diff_輪椅'].dt.total_seconds().mean()
輪椅_pre_time

np.float64(46.072916666666664)

In [16]:
routes_輪椅

,route_pair,Median_Traveling_Time,Median_Traveling_Time_輪椅,Time_diff_輪椅
104,"(PT, X光部)",0 days 00:03:46,0 days 00:03:25,-1 days +23:59:39
938,"(3C, X光部)",0 days 00:03:52.500000,0 days 00:05:02,0 days 00:01:09.500000
961,"(X光部, 門診部)",0 days 00:02:26,0 days 00:02:33,0 days 00:00:07
1077,"(3E, X光部)",0 days 00:02:54.500000,0 days 00:03:19,0 days 00:00:24.500000
1107,"(1B, X光部)",0 days 00:02:38,0 days 00:03:21.500000,0 days 00:00:43.500000
2686,"(3L, X光部)",0 days 00:06:12.250000,0 days 00:04:28,-1 days +23:58:15.750000
4027,"(3樓復, X光部)",0 days 00:03:36,0 days 00:05:57.500000,0 days 00:02:21.500000
4622,"(X光部, 日間)",0 days 00:02:53.500000,0 days 00:02:24,-1 days +23:59:30.500000
8023,"(6L, X光部)",0 days 00:12:19,0 days 00:06:44.500000,-1 days +23:54:25.500000
8261,"(3E, CDU)",0 days 00:02:24.250000,0 days 00:03:57,0 days 00:01:32.750000


In [17]:
輪床_data = clean_data1[(clean_data1['設備'] == '輪床') | (clean_data1['設備'] == '輪床O2')]
輪床_data = 輪床_data[輪床_data['主編號(序)'] == ' ']
輪床_data = 輪床_data[輪床_data['途經'] == ' ']
輪床_data = 輪床_data[輪床_data[' 從 '] != 輪床_data[' 往']]
輪床_data = 輪床_data[輪床_data['服務'].isin(LOW_OVERHEAD_SERVICES)]
comment = sorted(輪床_data['床號/備註'].unique(), key=sort_key)

輪床_data = 輪床_data[輪床_data['床號/備註'].isin(clean_comments)].copy()
輪床_data['Traveling_Time'] = 輪床_data['完成'] - 輪床_data['報到']
輪床_data['Traveling_Time'] = 輪床_data['Traveling_Time'].apply(format_timedelta_robust)
輪床_data['Traveling_Time'] = pd.to_timedelta(輪床_data['Traveling_Time'])
輪床_data = 輪床_data[輪床_data['Traveling_Time'] > pd.Timedelta(0)]

輪床_data['route_pair'] = 輪床_data.apply(
    lambda row: tuple(sorted([row[' 從 '], row[' 往']])), axis=1
)
median_per_route_輪床 = 輪床_data.groupby('route_pair')['Traveling_Time'].quantile(0.75)
輪床_data['Median_Traveling_Time'] = 輪床_data['route_pair'].map(median_per_route_輪床)
輪床_data

,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,...,設備,預定,下單,派遣,報到,完成,工友,Traveling_Time,route_pair,Median_Traveling_Time
3819,84351,,完成,即時,普通,S,送輕型物件,7H,A&D,,...,輪床,,2024-01-15 15:29:21,2024-01-15 16:27:59,2024-01-15 16:30:26,2024-01-15 16:38:33,3047,0 days 00:08:07,"(7H, A&D)",0 days 00:08:07
6447,87900,,完成,超緊急,普通,S,送X光,CDU,X光部,,...,輪床,,2024-01-25 09:14:42,2024-01-25 09:30:20,2024-01-25 09:32:58,2024-01-25 09:37:14,0071,0 days 00:04:16,"(CDU, X光部)",0 days 00:04:32.500000
7137,88814,,完成,即時,普通,S,送X光,7H,X光部,,...,輪床,,2024-01-27 10:41:38,2024-01-27 10:45:16,2024-01-27 10:50:28,2024-01-27 10:55:34,"0002,3027",0 days 00:05:06,"(7H, X光部)",0 days 00:08:56.250000
10598,94328,,完成,即時,預定,S,送3F X光,3H,3FXRAY,,...,輪床,2024-02-14 14:00:00,2024-02-14 09:20:17,2024-02-14 14:18:55,2024-02-14 14:26:30,2024-02-14 14:31:49,"0002,0023",0 days 00:05:19,"(3FXRAY, 3H)",0 days 00:05:43
10635,94370,,完成,即時,預定,S,送3F X光,3H,3FXRAY,,...,輪床,2024-02-14 14:00:00,2024-02-14 10:00:57,2024-02-14 14:14:34,2024-02-14 14:18:56,2024-02-14 14:22:38,"0071,3027",0 days 00:03:42,"(3FXRAY, 3H)",0 days 00:05:43
10745,94504,,完成,即時,預定,S,送3F X光,3H,3FXRAY,,...,輪床,2024-02-14 14:00:00,2024-02-14 12:32:12,2024-02-14 14:15:02,2024-02-14 14:15:59,2024-02-14 14:20:22,"3029,3047",0 days 00:04:23,"(3FXRAY, 3H)",0 days 00:05:43
11833,95983,,完成,即時,預定,S,送3F X光,3H,3FXRAY,,...,輪床,2024-02-19 14:00:00,2024-02-19 11:02:58,2024-02-19 14:12:37,2024-02-19 14:20:09,2024-02-19 14:24:26,"3026,3027",0 days 00:04:17,"(3FXRAY, 3H)",0 days 00:05:43
13221,97844,,完成,超緊急,普通,S,送3F X光,門診部,3FXRAY,,...,輪床,,2024-02-23 14:01:49,2024-02-23 14:11:11,2024-02-23 14:14:18,2024-02-23 14:15:43,0021,0 days 00:01:25,"(3FXRAY, 門診部)",0 days 00:04:50
13732,98606,,完成,即時,預定,S,送3F X光,3H,3FXRAY,,...,輪床,2024-02-27 14:00:00,2024-02-26 13:15:06,2024-02-27 14:13:35,2024-02-27 14:16:49,2024-02-27 14:21:07,"0002,0069",0 days 00:04:18,"(3FXRAY, 3H)",0 days 00:05:43
15391,895,,完成,即時,預定,S,送3F X光,3H,3FXRAY,,...,輪床,2024-03-04 14:15:00,2024-03-04 11:25:21,2024-03-04 14:18:54,2024-03-04 14:28:37,2024-03-04 14:33:07,"3047,3048",0 days 00:04:30,"(3FXRAY, 3H)",0 days 00:05:43


In [18]:
routes_輪床 = routes.copy()
routes_輪床['Median_Traveling_Time_輪床'] = routes_輪床['route_pair'].map(median_per_route_輪床)
routes_輪床 = routes_輪床[routes_輪床['Median_Traveling_Time_輪床'].notna()]
routes_輪床['Time_diff_輪床'] = routes_輪床['Median_Traveling_Time_輪床'] - routes_輪床['Median_Traveling_Time']
輪床_pre_time = routes_輪床['Time_diff_輪床'].dt.total_seconds().mean()
輪床_pre_time

np.float64(145.825)

In [19]:
routes_輪床

,route_pair,Median_Traveling_Time,Median_Traveling_Time_輪床,Time_diff_輪床
961,"(X光部, 門診部)",0 days 00:02:26,0 days 00:08:25,0 days 00:05:59
1083,"(1A, A&D)",0 days 00:07:02,0 days 00:08:14,0 days 00:01:12
2315,"(5L, A&D)",0 days 00:04:14,0 days 00:04:22,0 days 00:00:08
6031,"(3C, A&D)",0 days 00:08:52,0 days 00:09:14,0 days 00:00:22
11235,"(7H, A&D)",0 days 00:06:48,0 days 00:08:07,0 days 00:01:19
13246,"(3FXRAY, 門診部)",0 days 00:04:31,0 days 00:04:50,0 days 00:00:19
19175,"(3E, A&D)",0 days 00:08:17,0 days 00:14:58,0 days 00:06:41
21747,"(2A, X光部)",0 days 00:04:30,0 days 00:05:09,0 days 00:00:39
27319,"(1C, X光部)",0 days 00:02:55,0 days 00:06:25.250000,0 days 00:03:30.250000
29423,"(7H, X光部)",0 days 00:04:47.250000,0 days 00:08:56.250000,0 days 00:04:09


In [20]:
手推車_data = clean_data1[(clean_data1['設備'] == '手推車')]
手推車_data = 手推車_data[手推車_data['主編號(序)'] == ' ']
手推車_data = 手推車_data[手推車_data['途經'] == ' ']
手推車_data = 手推車_data[手推車_data[' 從 '] != 手推車_data[' 往']]
手推車_data = 手推車_data[手推車_data['服務'].isin(LOW_OVERHEAD_SERVICES)]
comment = sorted(手推車_data['床號/備註'].unique(), key=sort_key)

手推車_data = 手推車_data[手推車_data['床號/備註'].isin(clean_comments)].copy()
手推車_data['Traveling_Time'] = 手推車_data['完成'] - 手推車_data['報到']
手推車_data['Traveling_Time'] = 手推車_data['Traveling_Time'].apply(format_timedelta_robust)
手推車_data['Traveling_Time'] = pd.to_timedelta(手推車_data['Traveling_Time'])
手推車_data = 手推車_data[手推車_data['Traveling_Time'] > pd.Timedelta(0)]

手推車_data['route_pair'] = 手推車_data.apply(
    lambda row: tuple(sorted([row[' 從 '], row[' 往']])), axis=1
)
median_per_route_手推車 = 手推車_data.groupby('route_pair')['Traveling_Time'].quantile(0.75)
手推車_data['Median_Traveling_Time'] = 手推車_data['route_pair'].map(median_per_route_手推車)
手推車_data

,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,...,設備,預定,下單,派遣,報到,完成,工友,Traveling_Time,route_pair,Median_Traveling_Time
407,79765,,完成,即時,普通,S,送輕型物件,3FPOD,消毒房,,...,手推車,,2024-01-02 16:10:51,2024-01-02 17:07:34,2024-01-02 17:21:31,2024-01-02 17:26:10,3028,0 days 00:04:39,"(3FPOD, 消毒房)",0 days 00:09:03.750000
574,79991,,完成,即時,普通,S,送輕型物件,5L,6L,,...,手推車,,2024-01-03 11:27:39,2024-01-03 11:28:16,2024-01-03 11:34:07,2024-01-03 11:36:38,3022,0 days 00:02:31,"(5L, 6L)",0 days 00:02:31
767,80231,,完成,即時,普通,S,送輕型物件,3FPOD,消毒房,,...,手推車,,2024-01-03 15:52:46,2024-01-03 15:58:28,2024-01-03 16:11:18,2024-01-03 16:17:59,3028,0 days 00:06:41,"(3FPOD, 消毒房)",0 days 00:09:03.750000
1062,80622,,完成,即時,普通,S,送輕型物件,3FPOD,消毒房,,...,手推車,,2024-01-04 15:46:31,2024-01-04 15:46:48,2024-01-04 15:51:34,2024-01-04 15:57:08,3029,0 days 00:05:34,"(3FPOD, 消毒房)",0 days 00:09:03.750000
1427,81092,,完成,即時,普通,S,送輕型物件,3FPOD,消毒房,,...,手推車,,2024-01-05 15:54:44,2024-01-05 16:14:25,2024-01-05 16:23:49,2024-01-05 16:37:54,0021,0 days 00:14:05,"(3FPOD, 消毒房)",0 days 00:09:03.750000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83419,95964,,完成,即時,預定,S,送輕型物件,卸貨區,人事部,,...,手推車,2024-12-27 14:30:00,2024-12-27 11:38:05,2024-12-27 14:27:38,2024-12-27 14:33:21,2024-12-27 14:37:18,0037,0 days 00:03:57,"(人事部, 卸貨區)",0 days 00:03:57
83548,96130,,完成,即時,普通,S,送輕型物件,3FPOD,消毒房,,...,手推車,,2024-12-27 15:33:16,2024-12-27 15:39:07,2024-12-27 15:45:19,2024-12-27 15:51:47,3047,0 days 00:06:28,"(3FPOD, 消毒房)",0 days 00:09:03.750000
83875,96642,,完成,緊急,普通,S,送輕型物件,3C,1C,,...,手推車,,2024-12-30 12:00:20,2024-12-30 12:02:21,2024-12-30 12:08:30,2024-12-30 12:15:06,3048,0 days 00:06:36,"(1C, 3C)",0 days 00:08:52
84031,96849,,完成,緊急,普通,S,送輕型物件,3FPOD,消毒房,,...,手推車,,2024-12-30 16:08:35,2024-12-30 16:13:27,2024-12-30 16:16:10,2024-12-30 16:19:49,3026,0 days 00:03:39,"(3FPOD, 消毒房)",0 days 00:09:03.750000


In [21]:
routes_手推車 = routes.copy()
routes_手推車['Median_Traveling_Time_手推車'] = routes_手推車['route_pair'].map(median_per_route_手推車)
routes_手推車 = routes_手推車[routes_手推車['Median_Traveling_Time_手推車'].notna()]
routes_手推車['Time_diff'] = routes_手推車['Median_Traveling_Time_手推車'] - routes_手推車['Median_Traveling_Time']
手推車_pre_time = routes_手推車['Time_diff'].dt.total_seconds().mean()
手推車_pre_time

np.float64(182.05555555555554)

In [22]:
routes_手推車

,route_pair,Median_Traveling_Time,Median_Traveling_Time_手推車,Time_diff
34,"(7H, 藥房)",0 days 00:05:36,0 days 00:07:12,0 days 00:01:36
1554,"(2B, HIRD)",0 days 00:05:30.750000,0 days 00:04:14,-1 days +23:58:43.250000
1565,"(3E, 4D)",0 days 00:03:57.750000,0 days 00:10:58,0 days 00:07:00.250000
2395,"(3H, 7H)",0 days 00:08:26,0 days 00:13:40.500000,0 days 00:05:14.500000
4466,"(2A, 5H)",0 days 00:07:51.500000,0 days 00:08:00.250000,0 days 00:00:08.750000
4993,"(1C, 3C)",0 days 00:02:32.750000,0 days 00:08:52,0 days 00:06:19.250000
6402,"(3L, 6L)",0 days 00:08:17,0 days 00:17:10,0 days 00:08:53
6528,"(5L, 6L)",0 days 00:06:03.250000,0 days 00:02:31,-1 days +23:56:27.750000
7549,"(2A, 6L)",0 days 00:07:28.250000,0 days 00:19:33,0 days 00:12:04.750000
7725,"(6L, CDU)",0 days 00:07:09,0 days 00:05:47,-1 days +23:58:38


In [23]:
帶冰袋_data = clean_data1[(clean_data1['設備'] == '帶冰袋')]
帶冰袋_data = 帶冰袋_data[帶冰袋_data['主編號(序)'] == ' ']
帶冰袋_data = 帶冰袋_data[帶冰袋_data['途經'] == ' ']
帶冰袋_data = 帶冰袋_data[帶冰袋_data[' 從 '] != 帶冰袋_data[' 往']]
帶冰袋_data = 帶冰袋_data[帶冰袋_data['服務'].isin(LOW_OVERHEAD_SERVICES)]
comment = sorted(帶冰袋_data['床號/備註'].unique(), key=sort_key)

帶冰袋_data = 帶冰袋_data[帶冰袋_data['床號/備註'].isin(clean_comments)].copy()
帶冰袋_data['Traveling_Time'] = 帶冰袋_data['完成'] - 帶冰袋_data['報到']
帶冰袋_data['Traveling_Time'] = 帶冰袋_data['Traveling_Time'].apply(format_timedelta_robust)
帶冰袋_data['Traveling_Time'] = pd.to_timedelta(帶冰袋_data['Traveling_Time'])
帶冰袋_data = 帶冰袋_data[帶冰袋_data['Traveling_Time'] > pd.Timedelta(0)]

帶冰袋_data['route_pair'] = 帶冰袋_data.apply(
    lambda row: tuple(sorted([row[' 從 '], row[' 往']])), axis=1
)
median_per_route_帶冰袋 = 帶冰袋_data.groupby('route_pair')['Traveling_Time'].quantile(0.75)
帶冰袋_data['Median_Traveling_Time'] = 帶冰袋_data['route_pair'].map(median_per_route_帶冰袋)
帶冰袋_data

,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,...,設備,預定,下單,派遣,報到,完成,工友,Traveling_Time,route_pair,Median_Traveling_Time
2828,82972,,完成,緊急,普通,S,送標本,3C,化驗室,,...,帶冰袋,,2024-01-11 11:41:56,2024-01-11 11:42:17,2024-01-11 11:46:54,2024-01-11 11:48:03,0008,0 days 00:01:09,"(3C, 化驗室)",0 days 00:01:09
35068,28734,,完成,緊急,普通,S,送標本,5L,化驗室,,...,帶冰袋,,2024-05-30 12:07:34,2024-05-30 12:08:24,2024-05-30 12:16:09,2024-05-30 12:19:49,3029,0 days 00:03:40,"(5L, 化驗室)",0 days 00:03:40
70273,77779,,完成,緊急,普通,S,送藥,藥房,5H,,...,帶冰袋,,2024-10-31 09:07:57,2024-10-31 09:16:02,2024-10-31 09:17:54,2024-10-31 09:21:35,3023,0 days 00:03:41,"(5H, 藥房)",0 days 00:03:41


In [24]:
median_per_route_帶冰袋

route_pair
(3C, 化驗室)   0 days 00:01:09
(5H, 藥房)    0 days 00:03:41
(5L, 化驗室)   0 days 00:03:40
Name: Traveling_Time, dtype: timedelta64[ns]

In [25]:
routes_帶冰袋 = routes.copy()
routes_帶冰袋['Median_Traveling_Time_帶冰袋'] = routes_帶冰袋['route_pair'].map(median_per_route_帶冰袋)
routes_帶冰袋 = routes_帶冰袋[routes_帶冰袋['Median_Traveling_Time_帶冰袋'].notna()]
routes_帶冰袋['Time_diff'] = routes_帶冰袋['Median_Traveling_Time_帶冰袋'] - routes_帶冰袋['Median_Traveling_Time']
帶冰袋_pre_time = routes_帶冰袋['Time_diff'].dt.total_seconds().mean()
帶冰袋_pre_time

np.float64(-118.0)

In [26]:
routes_帶冰袋

,route_pair,Median_Traveling_Time,Median_Traveling_Time_帶冰袋,Time_diff
1060,"(5L, 化驗室)",0 days 00:08:35,0 days 00:03:40,-1 days +23:55:05
2647,"(5H, 藥房)",0 days 00:03:38.500000,0 days 00:03:41,0 days 00:00:02.500000
4548,"(3C, 化驗室)",0 days 00:02:10.500000,0 days 00:01:09,-1 days +23:58:58.500000


In [27]:
大班椅_data = clean_data1[(clean_data1['設備'] == '大班椅')]
大班椅_data = 大班椅_data[大班椅_data['主編號(序)'] == ' ']
大班椅_data = 大班椅_data[大班椅_data['途經'] == ' ']
大班椅_data = 大班椅_data[大班椅_data[' 從 '] != 大班椅_data[' 往']]
大班椅_data = 大班椅_data[大班椅_data['服務'].isin(LOW_OVERHEAD_SERVICES)]
comment = sorted(大班椅_data['床號/備註'].unique(), key=sort_key)

大班椅_data = 大班椅_data[大班椅_data['床號/備註'].isin(clean_comments)].copy()
大班椅_data['Traveling_Time'] = 大班椅_data['完成'] - 大班椅_data['報到']
大班椅_data['Traveling_Time'] = 大班椅_data['Traveling_Time'].apply(format_timedelta_robust)
大班椅_data['Traveling_Time'] = pd.to_timedelta(大班椅_data['Traveling_Time'])
大班椅_data = 大班椅_data[大班椅_data['Traveling_Time'] > pd.Timedelta(0)]

大班椅_data['route_pair'] = 大班椅_data.apply(
    lambda row: tuple(sorted([row[' 從 '], row[' 往']])), axis=1
)
median_per_route_大班椅 = 大班椅_data.groupby('route_pair')['Traveling_Time'].quantile(0.75)
大班椅_data['Median_Traveling_Time'] = 大班椅_data['route_pair'].map(median_per_route_大班椅)
大班椅_data

,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,...,設備,預定,下單,派遣,報到,完成,工友,Traveling_Time,route_pair,Median_Traveling_Time
36034,30067,,完成,即時,普通,S,送X光,1A,X光部,,...,大班椅,,2024-06-04 09:21:03,2024-06-04 09:41:57,2024-06-04 09:50:08,2024-06-04 09:54:49,0002,0 days 00:04:41,"(1A, X光部)",0 days 00:04:41


In [28]:
板車_data = clean_data1[(clean_data1['設備'] == '板車')]
板車_data = 板車_data[板車_data['主編號(序)'] == ' ']
板車_data = 板車_data[板車_data['途經'] == ' ']
板車_data = 板車_data[板車_data[' 從 '] != 板車_data[' 往']]
板車_data = 板車_data[板車_data['服務'].isin(LOW_OVERHEAD_SERVICES)]
comment = sorted(板車_data['床號/備註'].unique(), key=sort_key)

板車_data = 板車_data[板車_data['床號/備註'].isin(clean_comments)].copy()
板車_data['Traveling_Time'] = 板車_data['完成'] - 板車_data['報到']
板車_data['Traveling_Time'] = 板車_data['Traveling_Time'].apply(format_timedelta_robust)
板車_data['Traveling_Time'] = pd.to_timedelta(板車_data['Traveling_Time'])
板車_data = 板車_data[板車_data['Traveling_Time'] > pd.Timedelta(0)]

板車_data['route_pair'] = 板車_data.apply(
    lambda row: tuple(sorted([row[' 從 '], row[' 往']])), axis=1
)
median_per_route_板車 = 板車_data.groupby('route_pair')['Traveling_Time'].quantile(0.75)
板車_data['Median_Traveling_Time'] = 板車_data['route_pair'].map(median_per_route_板車)
板車_data

,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,...,設備,預定,下單,派遣,報到,完成,工友,Traveling_Time,route_pair,Median_Traveling_Time
3832,84368,,完成,即時,預定,S,送輕型物件,卸貨區,7H,,...,板車,2024-01-16 09:35:00,2024-01-15 15:50:58,2024-01-16 09:36:17,2024-01-16 09:36:20,2024-01-16 09:54:49,0073,0 days 00:18:29,"(7H, 卸貨區)",0 days 01:07:14
12401,96738,,完成,即時,預定,S,送輕型物件,卸貨區,7H,,...,板車,2024-02-21 09:30:00,2024-02-21 08:58:42,2024-02-21 09:27:55,2024-02-21 09:31:56,2024-02-21 10:55:25,0069,0 days 01:23:29,"(7H, 卸貨區)",0 days 01:07:14
48837,48100,,完成,即時,預定,S,送3F X光,3L,3FXRAY,,...,板車,2024-07-31 13:45:00,2024-07-31 10:30:56,2024-07-31 14:14:32,2024-07-31 14:16:05,2024-07-31 14:23:32,3027,0 days 00:07:27,"(3FXRAY, 3L)",0 days 00:07:27


In [29]:
routes_板車 = routes.copy()
routes_板車['Median_Traveling_Time_板車'] = routes_板車['route_pair'].map(median_per_route_板車)
routes_板車 = routes_板車[routes_板車['Median_Traveling_Time_板車'].notna()]
routes_板車['Time_diff'] = routes_板車['Median_Traveling_Time_板車'] - routes_板車['Median_Traveling_Time']
板車_pre_time = routes_板車['Time_diff'].dt.total_seconds().mean()
板車_pre_time

nan

In [30]:
routes_板車

,route_pair,Median_Traveling_Time,Median_Traveling_Time_板車,Time_diff


In [31]:
大床_data = clean_data1[(clean_data1['設備'] == '大床') | (clean_data1['設備'] == '大床O2')]
大床_data = 大床_data[大床_data['主編號(序)'] == ' ']
大床_data = 大床_data[大床_data['途經'] == ' ']
大床_data = 大床_data[大床_data[' 從 '] != 大床_data[' 往']]
大床_data = 大床_data[大床_data['服務'].isin(LOW_OVERHEAD_SERVICES)]
comment = sorted(大床_data['床號/備註'].unique(), key=sort_key)

大床_data = 大床_data[大床_data['床號/備註'].isin(clean_comments)].copy()
大床_data['Traveling_Time'] = 大床_data['完成'] - 大床_data['報到']
大床_data['Traveling_Time'] = 大床_data['Traveling_Time'].apply(format_timedelta_robust)
大床_data['Traveling_Time'] = pd.to_timedelta(大床_data['Traveling_Time'])
大床_data = 大床_data[大床_data['Traveling_Time'] > pd.Timedelta(0)]

大床_data['route_pair'] = 大床_data.apply(
    lambda row: tuple(sorted([row[' 從 '], row[' 往']])), axis=1
)
median_per_route_大床 = 大床_data.groupby('route_pair')['Traveling_Time'].quantile(0.75)
大床_data['Median_Traveling_Time'] = 大床_data['route_pair'].map(median_per_route_大床)
大床_data

,編號,主編號(序),狀態,優先,類別,種,服務,從,往,途經,...,設備,預定,下單,派遣,報到,完成,工友,Traveling_Time,route_pair,Median_Traveling_Time
0,79241,,完成,即時,預定,S,送3F X光,7H,3FXRAY,,...,大床,2024-01-02 14:00:00,2024-01-01 10:28:24,2024-01-02 14:35:48,2024-01-02 14:38:15,2024-01-02 14:44:57,"0030,0090",0 days 00:06:42,"(3FXRAY, 7H)",0 days 00:09:24
19,79303,,完成,即時,預定,S,送X光,4D,X光部,,...,大床,2024-01-02 09:15:00,2024-01-02 08:43:00,2024-01-02 09:47:44,2024-01-02 09:56:22,2024-01-02 10:05:21,"3022,3023",0 days 00:08:59,"(4D, X光部)",0 days 00:05:09
20,79304,,完成,即時,預定,S,送X光,4D,X光部,,...,大床,2024-01-02 09:15:00,2024-01-02 08:43:07,2024-01-02 09:56:29,2024-01-02 09:58:55,2024-01-02 10:02:29,"0035,0073",0 days 00:03:34,"(4D, X光部)",0 days 00:05:09
21,79305,,完成,超緊急,預定,S,送X光,4D,X光部,,...,大床,2024-01-02 09:15:00,2024-01-02 08:43:20,2024-01-02 10:09:29,2024-01-02 10:10:39,2024-01-02 10:13:01,"0002,3048",0 days 00:02:22,"(4D, X光部)",0 days 00:05:09
22,79306,,完成,即時,預定,S,送X光,4D,X光部,,...,大床,2024-01-02 09:20:00,2024-01-02 08:43:29,2024-01-02 10:03:14,2024-01-02 10:10:21,2024-01-02 10:13:34,"0035,0073",0 days 00:03:13,"(4D, X光部)",0 days 00:05:09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84253,97171,,完成,即時,預定,S,送3F X光,5H,3FXRAY,,...,大床,2024-12-31 14:15:00,2024-12-31 13:46:18,2024-12-31 14:16:58,2024-12-31 14:19:52,2024-12-31 14:26:55,"0061,3048",0 days 00:07:03,"(3FXRAY, 5H)",0 days 00:08:05.500000
84255,97174,,完成,即時,普通,S,送3F X光,3L,3FXRAY,,...,大床,,2024-12-31 13:57:01,2024-12-31 14:08:26,2024-12-31 14:12:46,2024-12-31 14:14:49,"0061,3048",0 days 00:02:03,"(3FXRAY, 3L)",0 days 00:02:44
84256,97175,,完成,即時,普通,S,送3F X光,3L,3FXRAY,,...,大床,,2024-12-31 13:57:14,2024-12-31 14:08:18,2024-12-31 14:09:17,2024-12-31 14:10:29,"0009,3026",0 days 00:01:12,"(3FXRAY, 3L)",0 days 00:02:44
84259,97180,,完成,緊急,普通,S,送X光,1B,3FXRAY,,...,大床,,2024-12-31 14:08:55,2024-12-31 14:19:52,2024-12-31 14:23:04,2024-12-31 14:33:30,3028,0 days 00:10:26,"(1B, 3FXRAY)",0 days 00:10:40


In [32]:
routes_大床 = routes.copy()
routes_大床['Median_Traveling_Time_大床'] = routes_大床['route_pair'].map(median_per_route_大床)
routes_大床 = routes_大床[routes_大床['Median_Traveling_Time_大床'].notna()]
routes_大床['Time_diff'] = routes_大床['Median_Traveling_Time_大床'] - routes_大床['Median_Traveling_Time']
大床_pre_time = routes_大床['Time_diff'].dt.total_seconds().mean()
大床_pre_time

np.float64(60.515625)

In [33]:
combined_unique = pd.concat([clean_data[' 從 '], clean_data[' 往']]).unique()
combined_unique

array(['7H', '5L', '抽血室', '3C', '6L', '1B', '3E', '化驗室', 'A&D', '4D',
       '1C', '3B', 'HIRD', '2A', '膳食部', '3H', 'X光部', '3L', '1A', '5H',
       'PT', 'CDU', '6H', 'PRC', '門診部', '2F矯形', '2B', '5F復康', '3FXRAY',
       '2F', '3FPOD', '日間', 'CNS', '標本室', '大堂', '消毒房', 'MED', '2FOT',
       '支援部', '卸貨區', '藥房', '被服房', '2FPT', '活動室', 'ICSC', '物料部', 'SDU',
       'ICT', '人事部', '太平間', '3FCP', '2FDOM', '行政部', '3樓復', '3F抽血', 'NSD',
       'P&PC', '3F 復', '家居', '疥瘡房', '6F復康', '1FMSW', '繳費處', '3FDIET',
       'ENQ', '1F演講', '工程部', '總務處', '陳笑風', '3F復康', '3FST', 'GF走廊', '專職2F',
       'PRO', 'OT', '財務部', '詢問處', '保安室', '電腦部', '3FAHS', 'PMM', '課室',
       '未知', '3B大門', '垃圾房', '圖書館', 'GF附', '院牧部', '飯堂'], dtype=object)

In [34]:
locations = sorted(combined_unique, key=sort_key)

In [35]:
df = pd.DataFrame('', index=locations, columns=locations)
np.fill_diagonal(df.values, 0)

# Fill with 無工具 (no-equipment) observed data — low-overhead services, >= 5 observations
for (loc1, loc2), med_time in median_per_route.items():
    df.at[loc1, loc2] = med_time
    df.at[loc2, loc1] = med_time

# Fill remaining empty cells with raw equipment travel times (no overhead adjustment).
# Priority order: 輪椅, 輪床, 手推車, 帶冰袋, 大班椅, 板車, 大床
for equip_medians in [
    median_per_route_輪椅,
    median_per_route_輪床,
    median_per_route_手推車,
    median_per_route_帶冰袋,
    median_per_route_大班椅,
    median_per_route_板車,
    median_per_route_大床,
]:
    for (loc1, loc2), med_time in equip_medians.items():
        if df.at[loc1, loc2] == '' or pd.isna(df.at[loc1, loc2]):
            df.at[loc1, loc2] = med_time
            df.at[loc2, loc1] = med_time

n = len(locations)
filled = sum(1 for l1 in locations for l2 in locations if l1 != l2 and df.at[l1, l2] != '')
print(f'After observed data: {filled}/{n*(n-1)} off-diagonal cells filled ({filled/n/(n-1):.1%})')

After observed data: 496/7832 off-diagonal cells filled (6.3%)


In [36]:
df

,1A,1B,1C,1FMSW,1F演講,2A,2B,2F,2FDOM,2FOT,...,行政部,被服房,詢問處,課室,財務部,門診部,院牧部,陳笑風,電腦部,飯堂
1A,0,,0 days 00:01:17,,,,,0 days 00:04:08.500000,0 days 00:07:26.500000,,...,,,,,,,,,,
1B,,0,,0 days 00:06:16.750000,,,,,,,...,,,,,,,,,,
1C,0 days 00:01:17,,0,,,,,,,,...,,,,,,,,,,
1FMSW,,0 days 00:06:16.750000,,0,,,,0 days 00:08:34.500000,,,...,,,,,,,,,,
1F演講,,,,,0,,,,,,...,0 days 00:12:34,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
門診部,,,,,,,,,,,...,,,,,,0,,,,
院牧部,,,,,,,,,,,...,0 days 00:07:57,,,,,,0,,,
陳笑風,,,,,,,,,,,...,,,,,,,,0,,
電腦部,,,,,,,,,,,...,,,,,,,,,0,


In [37]:
# Shortest-path imputation via Floyd-Warshall.
# For any (i, j) pair still empty, fills with the minimum-cost path through known pairs.

def cell_to_minutes(val):
    if isinstance(val, pd.Timedelta):
        return val.total_seconds() / 60
    if val == 0:
        return 0.0
    return None  # unknown / empty

n = len(locations)

# Build float matrix (None = unknown)
mat = [[cell_to_minutes(df.at[loc1, loc2]) for loc2 in locations] for loc1 in locations]

# Floyd-Warshall
for k in range(n):
    for i in range(n):
        if mat[i][k] is None:
            continue
        for j in range(n):
            if mat[k][j] is None:
                continue
            via = mat[i][k] + mat[k][j]
            if mat[i][j] is None or via < mat[i][j]:
                mat[i][j] = via

# Write imputed values back for cells that were empty
imputed = 0
for i, loc1 in enumerate(locations):
    for j, loc2 in enumerate(locations):
        if df.at[loc1, loc2] == '' and mat[i][j] is not None:
            df.at[loc1, loc2] = pd.Timedelta(minutes=mat[i][j])
            imputed += 1

filled_after = sum(1 for l1 in locations for l2 in locations if l1 != l2 and df.at[l1, l2] != '')
print(f'Imputed {imputed} additional cells via shortest-path')
print(f'Final matrix fill rate: {filled_after}/{n*(n-1)} = {filled_after/n/(n-1):.1%}')
df

Imputed 3410 additional cells via shortest-path
Final matrix fill rate: 3906/7832 = 49.9%


,1A,1B,1C,1FMSW,1F演講,2A,2B,2F,2FDOM,2FOT,...,行政部,被服房,詢問處,課室,財務部,門診部,院牧部,陳笑風,電腦部,飯堂
1A,0,0 days 00:04:31.250000,0 days 00:01:17,0 days 00:10:40.250000,0 days 00:23:25,0 days 00:04:04.750000,0 days 00:05:40.750000,0 days 00:04:08.500000,0 days 00:07:26.500000,0 days 00:12:56,...,0 days 00:10:51,0 days 00:07:16.750000,,,,0 days 00:05:34,0 days 00:18:48,,0 days 00:20:17,
1B,0 days 00:04:31.250000,0,0 days 00:05:33,0 days 00:06:16.750000,0 days 00:23:06.750000,0 days 00:03:46,0 days 00:07:05,0 days 00:08:39.750000,0 days 00:10:34.250000,0 days 00:09:49.999999999,...,0 days 00:10:32.750000,0 days 00:09:06.750000,,,,0 days 00:05:04,0 days 00:18:29.750000,,0 days 00:21:07.250000,
1C,0 days 00:01:17,0 days 00:05:33,0,0 days 00:10:00.750000,0 days 00:23:09,0 days 00:05:21.750000,0 days 00:05:55.750000,0 days 00:05:25.500000,0 days 00:08:43.500000,0 days 00:12:19,...,0 days 00:10:35,0 days 00:05:59.750000,,,,0 days 00:05:21,0 days 00:18:32,,0 days 00:19:00,
1FMSW,0 days 00:10:40.250000,0 days 00:06:16.750000,0 days 00:10:00.750000,0,0 days 00:24:49.750000,0 days 00:10:02.750000,0 days 00:11:15,0 days 00:08:34.500000,0 days 00:09:21.750000,0 days 00:12:02,...,0 days 00:12:15.750000,0 days 00:14:10.250000,,,,0 days 00:09:14,0 days 00:20:12.750000,,0 days 00:16:08.500000,
1F演講,0 days 00:23:25,0 days 00:23:06.750000,0 days 00:23:09,0 days 00:24:49.750000,0,0 days 00:23:09,0 days 00:22:57.750000,0 days 00:23:24,0 days 00:25:27.500000,0 days 00:29:25.750000,...,0 days 00:12:34,0 days 00:26:38.250000,,,,0 days 00:21:19.250000,0 days 00:20:31,,0 days 00:33:29,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
門診部,0 days 00:05:34,0 days 00:05:04,0 days 00:05:21,0 days 00:09:14,0 days 00:21:19.250000,0 days 00:04:48.750000,0 days 00:09:01.750000,0 days 00:09:42.500000,0 days 00:10:37.750000,0 days 00:08:48,...,0 days 00:08:45.250000,0 days 00:08:10.500000,,,,0,0 days 00:16:42.250000,,0 days 00:17:09.500000,
院牧部,0 days 00:18:48,0 days 00:18:29.750000,0 days 00:18:32,0 days 00:20:12.750000,0 days 00:20:31,0 days 00:18:32,0 days 00:18:20.750000,0 days 00:18:47,0 days 00:20:50.500000,0 days 00:24:48.750000,...,0 days 00:07:57,0 days 00:22:01.250000,,,,0 days 00:16:42.250000,0,,0 days 00:28:52,
陳笑風,,,,,,,,,,,...,,,,,,,,0,,
電腦部,0 days 00:20:17,0 days 00:21:07.250000,0 days 00:19:00,0 days 00:16:08.500000,0 days 00:33:29,0 days 00:19:20.250000,0 days 00:21:04,0 days 00:21:59,0 days 00:17:12.250000,0 days 00:22:18,...,0 days 00:20:55,0 days 00:24:59.750000,,,,0 days 00:17:09.500000,0 days 00:28:52,,0,


In [38]:
# Tier 4: Global conservative fallback
# For cells still empty after Floyd-Warshall, use the 90th percentile of all known travel times.
# Conservative by design — for unknown routes, assume near-worst-case observed time.
# This is appropriate for a 15-min KPI where underestimating is more costly than overestimating.
# Note: floor-based imputation (Tier 4b) can replace this for known-floor pairs in a future step.

known_times_min = [
    df.at[loc1, loc2].total_seconds() / 60
    for loc1 in locations for loc2 in locations
    if loc1 != loc2 and isinstance(df.at[loc1, loc2], pd.Timedelta)
]
p90_fallback = pd.Timedelta(minutes=pd.Series(known_times_min).quantile(0.90))
print(f'Global fallback value (p90 of known times): {p90_fallback}')

fallback_imputed = 0
for loc1 in locations:
    for loc2 in locations:
        if loc1 != loc2 and df.at[loc1, loc2] == '':
            df.at[loc1, loc2] = p90_fallback
            fallback_imputed += 1

n = len(locations)
total_filled = sum(1 for l1 in locations for l2 in locations if l1 != l2 and df.at[l1, l2] != '')
print(f'Fallback imputed: {fallback_imputed} cells')
print(f'Final matrix fill rate: {total_filled}/{n*(n-1)} = {total_filled/n/(n-1):.1%}')

Global fallback value (p90 of known times): 0 days 00:27:34.250000


Fallback imputed: 3926 cells
Final matrix fill rate: 7832/7832 = 100.0%


In [39]:
def to_time_string(value):
    """
    Convert a cell to a string:
    - NaN/None → "" (empty string)
    - Timedelta → "HH:MM:SS" (padded with zeros)
    - 0 (diagonal) → "00:00:00" (optional – remove if you prefer "0")
    - any other number → str(value) (e.g., "0")
    - any other type → str(value)
    """
    if pd.isnull(value):
        return ""                     # empty cell in Excel
    if isinstance(value, pd.Timedelta):
        total_seconds = int(value.total_seconds())
        hours, remainder = divmod(total_seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    # Optional: treat exact 0 as "00:00:00" for consistency
    if value == 0:
        return "00:00:00"
    return str(value)

# Apply the function to every cell using .map()
df_string = df.map(to_time_string)

# Save to Excel (index=True keeps row labels)
df_string.to_excel('travel_times.xlsx', index=True)

In [40]:
output_path = '/Users/adam/Projects/FYP/travel_times.xlsx'
with pd.ExcelWriter(output_path) as writer:
    df_string.to_excel(writer, sheet_name='Travel_Times_P75')
    stats_df.to_excel(writer, sheet_name='Route_Stats', index=False)
print(f'Saved to {output_path}')
print(f'  Sheet 1 (Travel_Times_P75): {len(df_string)}x{len(df_string.columns)} matrix')
print(f'  Sheet 2 (Route_Stats): {len(stats_df)} observed route pairs')

Saved to /Users/adam/Projects/FYP/travel_times.xlsx
  Sheet 1 (Travel_Times_P75): 89x89 matrix
  Sheet 2 (Route_Stats): 177 observed route pairs


In [41]:
df_string

,1A,1B,1C,1FMSW,1F演講,2A,2B,2F,2FDOM,2FOT,...,行政部,被服房,詢問處,課室,財務部,門診部,院牧部,陳笑風,電腦部,飯堂
1A,00:00:00,00:04:31,00:01:17,00:10:40,00:23:25,00:04:04,00:05:40,00:04:08,00:07:26,00:12:56,...,00:10:51,00:07:16,00:27:34,00:27:34,00:27:34,00:05:34,00:18:48,00:27:34,00:20:17,00:27:34
1B,00:04:31,00:00:00,00:05:33,00:06:16,00:23:06,00:03:46,00:07:05,00:08:39,00:10:34,00:09:49,...,00:10:32,00:09:06,00:27:34,00:27:34,00:27:34,00:05:04,00:18:29,00:27:34,00:21:07,00:27:34
1C,00:01:17,00:05:33,00:00:00,00:10:00,00:23:09,00:05:21,00:05:55,00:05:25,00:08:43,00:12:19,...,00:10:35,00:05:59,00:27:34,00:27:34,00:27:34,00:05:21,00:18:32,00:27:34,00:19:00,00:27:34
1FMSW,00:10:40,00:06:16,00:10:00,00:00:00,00:24:49,00:10:02,00:11:15,00:08:34,00:09:21,00:12:02,...,00:12:15,00:14:10,00:27:34,00:27:34,00:27:34,00:09:14,00:20:12,00:27:34,00:16:08,00:27:34
1F演講,00:23:25,00:23:06,00:23:09,00:24:49,00:00:00,00:23:09,00:22:57,00:23:24,00:25:27,00:29:25,...,00:12:34,00:26:38,00:27:34,00:27:34,00:27:34,00:21:19,00:20:31,00:27:34,00:33:29,00:27:34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
門診部,00:05:34,00:05:04,00:05:21,00:09:14,00:21:19,00:04:48,00:09:01,00:09:42,00:10:37,00:08:48,...,00:08:45,00:08:10,00:27:34,00:27:34,00:27:34,00:00:00,00:16:42,00:27:34,00:17:09,00:27:34
院牧部,00:18:48,00:18:29,00:18:32,00:20:12,00:20:31,00:18:32,00:18:20,00:18:47,00:20:50,00:24:48,...,00:07:57,00:22:01,00:27:34,00:27:34,00:27:34,00:16:42,00:00:00,00:27:34,00:28:52,00:27:34
陳笑風,00:27:34,00:27:34,00:27:34,00:27:34,00:27:34,00:27:34,00:27:34,00:27:34,00:27:34,00:27:34,...,00:27:34,00:27:34,00:27:34,00:27:34,00:27:34,00:27:34,00:27:34,00:00:00,00:27:34,00:27:34
電腦部,00:20:17,00:21:07,00:19:00,00:16:08,00:33:29,00:19:20,00:21:04,00:21:59,00:17:12,00:22:18,...,00:20:55,00:24:59,00:27:34,00:27:34,00:27:34,00:17:09,00:28:52,00:27:34,00:00:00,00:27:34


# Summary: What Was Wrong and What Was Fixed

## Problems with the Original Methodology

### 1. Service filter was too broad
The original filter included services where the porter does more than just walk between two points — patient transfers (`送病人`, `送入院`), round-trips (`雪藥來回`), multi-stop tasks (`送空瓶/箱`), and tasks with significant wait time at the destination (`文件簽收`, `抽血急標本`). These inflated measured travel times by including overhead that is not pure walking time.

**Fix:** Defined `LOW_OVERHEAD_SERVICES` — a whitelist of pure object-delivery services where the porter drops off and leaves immediately (`送標本`, `送文件`, `送X光`, etc.).

### 2. Equipment adjustment produced negative travel times
The original Cell 42 subtracted an estimated `prep_time` offset from equipment-category times to approximate a "no-equipment" baseline. This offset was derived from a noisy mean difference and frequently pushed values below zero — which is physically impossible.

**Fix:** Removed the adjustment entirely. Equipment times are used as direct fallback values for route pairs not covered by `無工具` data. Raw equipment times are a slight overestimate due to handling overhead, but are always positive and directionally correct.

### 3. No minimum sample size
Route pairs with only 1–2 observations were included, making the p75 estimate statistically meaningless (a single outlier could dominate).

**Fix:** Required ≥ 5 observations per route pair. This reduced valid pairs but each estimate is now based on a meaningful sample.

### 4. Matrix was 85% empty with no imputation
The original notebook left ~7,400 of 7,832 cells blank. An OR solver given a sparse matrix either crashes or silently treats missing routes as unreachable.

**Fix:** Four-tier fill strategy:
1. **Tier 1 — 無工具 observed p75** (primary, cleanest data): ~6% fill
2. **Tier 2 — Equipment-category observed p75** (fallback for uncovered pairs): ~44% fill
3. **Tier 3 — Floyd-Warshall shortest-path** (fills gaps via A→B + B→C chaining): ~50% fill
4. **Tier 4 — Global p90 conservative fallback** (flat worst-case for all remaining unknowns): 100% fill

The Tier 4 fallback uses the 90th percentile of all known travel times. For routes with no data and no chain, this is deliberately conservative — for a 15-min KPI, overestimating is safer than underestimating.

**Note on `完成 - 報到`:** This is travel time + brief pickup/dropoff overhead, not pure walking time. For `LOW_OVERHEAD_SERVICES` (specimen/document deliveries), the overhead is seconds. When integrating with OR-Tools, use Dij for arc costs and do not add separate service time for these services, to avoid double-counting.

### 5. Used median instead of a robust conservative estimate
Given a median standard deviation of **7.5 minutes** per route pair, the median will be beaten ~50% of the time.

**Fix:** Switched to the **75th percentile** as the Dij input. The solver's planned travel times will be met or beaten ~75% of the time, building a practical buffer without requiring a stochastic solver.

---

## Recommended Next Steps

### Task 1 — Integrate the matrix into `porter-dispatch-prototype`

The current prototype hardcodes a small `TRAVEL_TIME_MATRIX` dict of ~9 locations. Replace it with `travel_times.xlsx` (Sheet 1). Load it at startup:

```python
import pandas as pd

_df = pd.read_excel('travel_times.xlsx', sheet_name='Travel_Times_P75', index_col=0)
TRAVEL_TIME_MATRIX = {}
for loc1 in _df.index:
    for loc2 in _df.columns:
        val = _df.at[loc1, loc2]
        if val not in ('', '00:00:00'):
            h, m, s = map(int, val.split(':'))
            TRAVEL_TIME_MATRIX[(loc1, loc2)] = h * 60 + m + s / 60  # minutes
```

The `find_best_porter()` function already looks up `TRAVEL_TIME_MATRIX[(porter.location, task.origin)]` — no other changes needed for the greedy fallback.

### Task 2 — Replace greedy assignment with Google OR-Tools

The greedy algorithm in `find_best_porter()` assigns the nearest available porter one task at a time. This is suboptimal when multiple tasks and porters are available simultaneously. **Google OR-Tools** (Vehicle Routing Problem solver) considers all pending tasks together and finds a globally optimal assignment.

Install: `pip install ortools`

Key OR-Tools concepts for this system:
- **Routing model**: each porter is a "vehicle", each task is a "delivery node"
- **Arc costs**: use `TRAVEL_TIME_MATRIX` as the distance callback
- **Time windows**: add a time window constraint per task to enforce the 15-minute KPI
- **Task batching**: OR-Tools natively supports assigning multiple tasks per porter per route — this matches how HHH already manually batches tasks
- **Service time**: for `LOW_OVERHEAD_SERVICES`, Dij already includes pickup/dropoff overhead — do not add separate service time for these to avoid double-counting

A minimal integration point: call OR-Tools at each dispatch event instead of `find_best_porter()`, passing the current porter states and all queued + new tasks.

### Task 3 — Improve Tier 4 fallback with floor-based estimates (optional)

The current Tier 4 fallback uses a single flat p90 value for all unknown pairs. A floor-based model would give better estimates for cross-floor vs. same-floor pairs:
- 35 of 89 locations have floors inferrable from their names (e.g. `7H` → floor 7)
- The remaining 54 (e.g. `化驗室`, `A&D`, `HIRD`) require the hospital floor plan
- Fit `travel_time ~ base + per_floor_cost × |floor_diff|` from observed cross-floor pairs to calibrate constants empirically